# Home Assignment
## Two molecules a graph model cannot tell apart



---

### The chemistry

Compare the carbon skeletons of two hydrocarbons, hydrogens suppressed throughout.

```
        A: cyclohexane skeleton              B: two cyclopropane skeletons

               0                                   0            3
            /     \                               / \          / \
           1       5                             1---2        4---5
           |       |
           2       4
            \     /
               3
```

Both graphs have **six carbon atoms** and **six carbon-carbon bonds**, and every atom
carries the same initial label, `"C"`. Chemically they could hardly be more different:
one is a strain-free chair, the other carries roughly 115 kJ/mol of ring strain **per ring**.

Your task is to establish exactly what a standard message-passing model can and cannot
perceive here, and then to fix it.



In [14]:

import numpy as np
np.set_printoptions(precision=4, suppress=True)

def show(name, A):
    """Print an adjacency matrix with its degree column."""
    print(f'{name}   (degrees on the right)')
    for i, row in enumerate(A.astype(int)):
        print('  ' + ' '.join(str(v) for v in row) + f'   | d_{i} = {int(row.sum())}')
    print(f'  bonds = {int(A.sum() // 2)},  atoms = {A.shape[0]}')
    print()


---
## Task 1. Write both adjacency matrices  &nbsp;&nbsp;

Fill in `A_A` and `A_B` using the atom labels in the diagram above.

Reminders:
- The adjacency matrix is $A_{ij}=1$ when atoms $i$ and $j$ are bonded, and $0$ otherwise.
- A molecular graph is undirected, so $A$ must be **symmetric**.
- There are no self-bonds, so the diagonal is zero.
- In **B** the two rings are *not* connected to each other. Atoms 0,1,2 form one ring
  and atoms 3,4,5 the other.


In [15]:
# Adjacency matrix for the six-membered ring
adj_a = np.zeros((6, 6))
ring_a_edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0)]
for left, right in ring_a_edges:
    adj_a[left, right] = 1
    adj_a[right, left] = 1

# Adjacency matrix for the two separate three-membered rings
adj_b = np.zeros((6, 6))
ring_b_edges = [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)]
for left, right in ring_b_edges:
    adj_b[left, right] = 1
    adj_b[right, left] = 1

show_graph("A  (cyclohexane skeleton)", adj_a)
show_graph("B  (two cyclopropane skeletons)", adj_b)


A  (cyclohexane skeleton) (degrees on the right)
  0 1 0 0 0 1   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  0 1 0 1 0 0   | d_2 = 2
  0 0 1 0 1 0   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  1 0 0 0 1 0   | d_5 = 2
  bonds = 6, atoms = 6

B  (two cyclopropane skeletons) (degrees on the right)
  0 1 1 0 0 0   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  1 1 0 0 0 0   | d_2 = 2
  0 0 0 0 1 1   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  0 0 0 1 1 0   | d_5 = 2
  bonds = 6, atoms = 6



In [16]:
# Basic checks for the two adjacency matrices
for label, matrix in [("A", adj_a), ("B", adj_b)]:
    assert matrix.shape == (6, 6), f"{label}: must be 6x6"
    assert np.allclose(matrix, matrix.T), f"{label}: must be symmetric"
    assert np.all((matrix == 0) | (matrix == 1)), f"{label}: entries must be 0 or 1"
    assert np.allclose(np.diag(matrix), 0), f"{label}: diagonal must be zero"
    assert matrix.sum() / 2 == 6, f"{label}: must have exactly 6 bonds"

assert not np.allclose(adj_a, adj_b), "A and B must be different matrices"
print("Task 1 structural checks passed.")
print("Degree sequence A:", np.sort(adj_a.sum(axis=1)).astype(int))
print("Degree sequence B:", np.sort(adj_b.sum(axis=1)).astype(int))


Task 1 structural checks passed.
Degree sequence A: [2 2 2 2 2 2]
Degree sequence B: [2 2 2 2 2 2]


### YOUR ANSWER (Task 1)

For A, each carbon is connected to two other carbons, so all the degrees are 2:
`[2, 2, 2, 2, 2, 2]`.

The same is true for B. Even though B is made of two separate rings, every atom still has two carbon neighbours, so its degree list is also `[2, 2, 2, 2, 2, 2]`.

The degree just tells us how many atoms are directly bonded to a carbon. It does not tell us whether those bonds are part of one big ring or two smaller rings.


---
## Task 2. Colour refinement  &nbsp;&nbsp;

**Do this by hand first, on paper.** The code is to check your hand work, not to replace it.

The 1-WL refinement rule is

$$c^{(k+1)}_i \;=\; \mathrm{HASH}\Big(c^{(k)}_i,\ \{\!\{\,c^{(k)}_j : j \in \mathcal{N}(i)\,\}\!\}\Big)$$

where $\{\!\{\cdot\}\!\}$ is a **multiset** (repetition matters, order does not) and
$\mathrm{HASH}$ assigns a fresh integer to each distinct signature it has seen.

Two graphs are declared **distinguishable** if at some round their *multisets of colours*
differ.

Complete the function below.


In [17]:
def wl_round(adjacency, colours):
    """Perform one round of 1-WL colour refinement."""
    signatures = []

    for atom, colour in enumerate(colours):
        nearby = tuple(
            sorted(colours[neighbour]
                   for neighbour in range(len(colours))
                   if adjacency[atom, neighbour] != 0)
        )
        signatures.append((colour, nearby))

    # Give equal signatures equal numbers in a repeatable order.
    colour_ids = {
        signature: number
        for number, signature in enumerate(sorted(set(signatures), key=str))
    }
    return [colour_ids[signature] for signature in signatures]


### Validate your implementation on a case with a known answer

Before trusting `wl_round` on A and B, test it on a pair where the answer is already known:
the carbon skeletons of **n-butane** (a chain) and **isobutane** (a central carbon with
three neighbours).

These two *are* distinguishable, and refinement should separate them after one round,
because their degree sequences differ.


In [18]:
# A small test case where the degree pattern is different
adj_nbutane = np.array([
    [0, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
], dtype=float)

adj_isobutane = np.array([
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 0, 0],
    [1, 0, 0, 0],
], dtype=float)

butane_colours = wl_round(adj_nbutane, [0] * 4)
isobutane_colours = wl_round(adj_isobutane, [0] * 4)
print("n-butane colours after 1 round:", sorted(butane_colours))
print("isobutane colours after 1 round:", sorted(isobutane_colours))

assert sorted(butane_colours) != sorted(isobutane_colours), (
    "wl_round should separate n-butane from isobutane. "
    "Check that neighbour colours are treated as a multiset."
)
print("\nValidation passed: wl_round behaves correctly on a known case.")


n-butane colours after 1 round: [0, 0, 1, 1]
isobutane colours after 1 round: [0, 1, 1, 1]

Validation passed: wl_round behaves correctly on a known case.


In [19]:
colours_a = [0] * 6
colours_b = [0] * 6

print(f"{'round':<7}{'colours of A':<22}{'multiset A':<18}"
      f"{'colours of B':<22}{'multiset B'}")
print("-" * 92)

for round_number in range(3):
    print(f"{round_number:<7}{str(colours_a):<22}{str(sorted(colours_a)):<18}"
          f"{str(colours_b):<22}{str(sorted(colours_b))}")
    if round_number < 2:
        colours_a = wl_round(adj_a, colours_a)
        colours_b = wl_round(adj_b, colours_b)

print()
print("colour multisets identical at every round?",
      sorted(colours_a) == sorted(colours_b))


round  colours of A          multiset A        colours of B          multiset B
--------------------------------------------------------------------------------------------
0      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
1      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
2      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]

colour multisets identical at every round? True


### YOUR ANSWER (Task 2)

**Round 0:** Both graphs start with colour 0 on every atom. Therefore both colour lists are
`[0, 0, 0, 0, 0, 0]`.

**Round 1:** Every atom sees two neighbours, and both of those neighbours have colour 0. So every atom gets the same signature, `(0, (0, 0))`. The result is still six zeros for both A and B.

**Round 2:** Nothing changes. Every atom still has colour 0 and still sees two atoms with colour 0. Again, both graphs have the same colour multiset.

So the WL refinement cannot tell these graphs apart. This happens because it only sees the same local pattern around each atom, not how the separate local patterns are connected globally.


---
## Task 3. State the conclusion  &nbsp;&nbsp;

You now know what refinement does to A and B.


### YOUR ANSWER (Task 3)

No, a standard message-passing model cannot distinguish A and B from these starting features.

Both graphs have six atoms, every atom is labelled the same, and every atom has two neighbours. This means that each atom receives the same kind of message at every layer. Graph A is one six-atom cycle, while graph B is two separate three-atom cycles, but that larger-scale difference is invisible to the usual local updates.

Making the network wider, deeper, or training it for longer would not fix this particular problem. The missing information has to be added in some other way.


---
## Task 4. Find a discriminating invariant  &nbsp;&nbsp;

Refinement failed. Something else must succeed, because the two graphs genuinely differ.

Recall the walk-counting theorem: $(A^k)_{ij}$ is the number of walks of length exactly
$k$ from atom $i$ to atom $j$. Therefore $\mathrm{tr}(A^k) = \sum_i (A^k)_{ii}$ counts
**closed** walks of length $k$, that is walks that return to where they started.


In [20]:
print(f"{'k':<5}{'tr(adj_a^k)':>12}{'tr(adj_b^k)':>12}   separates?")
print("-" * 45)

for power in (2, 3, 6):
    trace_a = np.trace(np.linalg.matrix_power(adj_a, power))
    trace_b = np.trace(np.linalg.matrix_power(adj_b, power))
    answer = "YES" if trace_a != trace_b else "no"
    print(f"{power:<5}{trace_a:>12.0f}{trace_b:>12.0f}   {answer}")


k     tr(adj_a^k) tr(adj_b^k)   separates?
---------------------------------------------
2              12          12   no
3               0          12   YES
6             132         132   no


In [21]:
# The spectra are another way to compare the two adjacency matrices.
print("eigenvalues of adj_a:", np.sort(np.linalg.eigvalsh(adj_a))[::-1])
print("eigenvalues of adj_b:", np.sort(np.linalg.eigvalsh(adj_b))[::-1])


eigenvalues of adj_a: [ 2.  1.  1. -1. -1. -2.]
eigenvalues of adj_b: [ 2.  2. -1. -1. -1. -1.]


### YOUR ANSWER (Task 4)

**(a)** The first useful value is `k = 3`:

- `tr(A_A^3) = 0`
- `tr(A_B^3) = 12`

So this trace separates the two graphs.

**(b)** A length-3 closed walk is a trip around a triangle. A has no triangles, so its trace is zero. B has two triangles. Each triangle can be started at any of its three atoms and followed in either direction, giving $3 \times 2 = 6$ walks per triangle. The two triangles give 12 altogether.

**(c)** For `k = 2`, both traces are 12 because both graphs have six edges, and each edge can be crossed and immediately crossed back in two directions. For `k = 6`, both traces happen to be 132, so that value does not separate them. In A, some walks go around the six-membered ring; in B, walks can go around the smaller rings more than once.

**(d)** In general, `tr(A^2)` is twice the number of edges. Since both graphs have six edges, both must have the same value, 12.


---
## Task 5. Propose a fix and defend it  &nbsp;&nbsp;

Computing $\mathrm{tr}(A^k)$ costs $O(n^3)$ and does not transfer cleanly between
molecules of different size. A cheaper repair is to give each atom an extra **input
feature** that already distinguishes the two cases, so that the model separates them at
layer zero without any change to the architecture.

Implement your chosen feature below.


In [22]:
def extra_feature(adjacency):
    """Return the size of the connected component for each atom."""
    number_of_atoms = adjacency.shape[0]
    ring_sizes = np.zeros(number_of_atoms)
    visited = np.zeros(number_of_atoms, dtype=bool)

    for first_atom in range(number_of_atoms):
        if visited[first_atom]:
            continue

        stack = [first_atom]
        component_atoms = []
        visited[first_atom] = True

        while stack:
            current_atom = stack.pop()
            component_atoms.append(current_atom)
            neighbours = np.flatnonzero(adjacency[current_atom])
            for neighbour in neighbours:
                if not visited[neighbour]:
                    visited[neighbour] = True
                    stack.append(int(neighbour))

        ring_sizes[component_atoms] = len(component_atoms)

    return ring_sizes


feature_a = extra_feature(adj_a)
feature_b = extra_feature(adj_b)
print("feature on A:", feature_a)
print("feature on B:", feature_b)


feature on A: [6. 6. 6. 6. 6. 6.]
feature on B: [3. 3. 3. 3. 3. 3.]


In [23]:
assert not np.allclose(np.sort(feature_a), np.sort(feature_b)), (
    "The feature has the same values on A and B, so it cannot separate them."
)
print("The feature separates A from B at the input layer.")

# Relabelling atoms should not change the feature values.
permutation = np.random.default_rng(0).permutation(6)
adj_a_relabelled = adj_a[np.ix_(permutation, permutation)]
assert np.allclose(
    np.sort(extra_feature(adj_a_relabelled)), np.sort(feature_a)
), "The feature must be unchanged by relabelling the atoms."
print("The feature is unchanged by relabelling the atoms, as required.")


The feature separates A from B at the input layer.
The feature is unchanged by relabelling the atoms, as required.


### YOUR ANSWER (Task 5)

**(a)** I used the size of the ring or connected component containing each atom as the extra feature.

For A, the feature values are `[6, 6, 6, 6, 6, 6]`, since all six atoms are in one ring. For B, the values are `[3, 3, 3, 3, 3, 3]`, since its atoms are split between two three-membered rings. This makes the two graphs different right at the input layer.

**(b)** For these examples, I can find the connected components with DFS or BFS and assign the component size to each atom. That takes $O(n + m)$ time, where $n$ is the number of atoms and $m$ is the number of bonds. Ring information is also something that chemistry software can calculate from a molecular graph.

**(c)** Ring size is useful for predicting ring strain. A three-membered ring, such as cyclopropane, has much more angle and torsional strain than a six-membered cyclohexane ring.

**(d)** My main takeaway is that adding more message-passing layers would not solve a limitation that is caused by the input representation. Adding a meaningful feature that contains the missing ring information is a better fix here.


---
## Before you submit

Run the cell below. It confirms only that the notebook executes; it does not mark your
prose answers.


In [24]:
checks = {
    "Task 1: adjacency matrices built": (
        adj_a.sum() == 12
        and adj_b.sum() == 12
        and not np.allclose(adj_a, adj_b)
    ),
    "Task 2: wl_round implemented": (
        sorted(wl_round(adj_nbutane, [0] * 4))
        != sorted(wl_round(adj_isobutane, [0] * 4))
    ),
    "Task 5: extra_feature implemented": not np.allclose(
        extra_feature(adj_a), 0
    ),
}

for description, passed in checks.items():
    print(("  OK   " if passed else "  TODO ") + description)

print()
print("Remember: Kernel > Restart & Run All, then save with all output visible.")


  OK   Task 1: adjacency matrices built
  OK   Task 2: wl_round implemented
  OK   Task 5: extra_feature implemented

Remember: Kernel > Restart & Run All, then save with all output visible.
